# Tutorial 11: Agent Skills — Modular Knowledge Extension

## Learning Objectives
After completing this notebook, you will be able to:
- Understand the concept of Agent Skills and the **Progressive Disclosure pattern**
- Learn how to define skills using `SKILL.md` files
- Grant skills to agents using `SkillsProvider`
- Observe the 3-stage operation of skills: **Advertise → Load → Read Resources**
- Build agents with multiple skills

## Key Concepts

### What are Agent Skills?

**Agent Skills** are modular knowledge packages based on the [Agent Skills specification](https://agentskills.io/).  
Instead of stuffing everything into an agent's instructions, skills allow **loading only the knowledge needed, when it's needed**.

### Why are Skills Needed?

| Traditional Approach | Agent Skills |
|---|---|
| All domain knowledge in instructions | Only name + description advertised (~100 tokens/skill) |
| Consumes context window | Details loaded on demand |
| Doesn't scale (breaks as knowledge grows) | Scales by simply adding skills |
| Updating knowledge = code changes | Just update the `SKILL.md` file |

### Progressive Disclosure Pattern

```
1. Advertise
   └─ Inject skill name + description into system prompt (lightweight)

2. Load
   └─ Agent calls load_skill tool → retrieves SKILL.md body

3. Read Resources
   └─ Agent calls read_skill_resource tool to retrieve FAQ/templates, etc.
```

### Skill Directory Structure

```
skills/
├── travel-planner/
│   ├── SKILL.md              ← Main instruction file (YAML frontmatter + body)
│   ├── references/
│   │   └── TRAVEL_FAQ.md     ← Reference documents
│   └── assets/
│       └── itinerary-template.md  ← Templates
└── code-reviewer/
    ├── SKILL.md
    └── references/
        └── CHECKLIST.md
```

### SKILL.md Format

```markdown
---
name: travel-planner
description: Provides guidance on travel planning and booking.
---

# Body (detailed instructions for the agent)
Write detailed domain knowledge here...
Reference: [references/FAQ.md](references/FAQ.md)
Template: [assets/template.md](assets/template.md)
```

---

**Prerequisites for this tutorial**: `agent-framework >= 1.0.0rc2` (Agent Skills were newly added in rc2)


## Step 1: Setup and Imports


In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

from agent_framework import Agent, SkillsProvider, Message
from agent_framework.azure import AzureOpenAIChatClient

# Load environment variables
load_dotenv(override=True)

print("✅ Import successful!")
print("📦 Classes used for Agent Skills:")
print("   - SkillsProvider: File-based skill provider")
print("   - Agent: Agent with skills")


In [ ]:
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === Authentication Method Selection ===
# True: API Key authentication, False: DefaultAzureCredential authentication
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 Authentication method: API Key")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # The framework auto-reads the AZURE_OPENAI_API_KEY environment variable
    # and prioritizes it over credential, so explicitly remove it
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 Authentication method: DefaultAzureCredential")

print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}")
print(f"Deployment: {azure_deployment}")


## Step 2: OpenTelemetry Tracer Setup

Enable tracing to visualize the internal operations of Agent Skills
(`load_skill` / `read_skill_resource` tool calls).

Jaeger UI: [http://localhost:16686](http://localhost:16686)


In [ ]:
from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")
os.environ.setdefault("OTEL_METRICS_EXPORTER", "none")  # Disable Metrics (change as needed)

configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")


## Step 3: Inspect the Skill Files

First, let's examine the contents of the skill files included with this workshop.


In [ ]:
# Skill directory location
skills_dir = Path("skills")

print(f"📂 Skills directory: {skills_dir.absolute()}")
print()

# Display directory structure
for skill_dir in sorted(skills_dir.iterdir()):
    if skill_dir.is_dir():
        print(f"📁 {skill_dir.name}/")
        for item in sorted(skill_dir.rglob("*")):
            if item.is_file():
                relative = item.relative_to(skill_dir)
                indent = "  " * (len(relative.parts) - 1)
                print(f"   {indent}📄 {relative}")


In [ ]:
# Read travel-planner's SKILL.md
skill_file = skills_dir / "travel-planner" / "SKILL.md"
content = skill_file.read_text(encoding="utf-8")

print("=" * 70)
print("📄 travel-planner/SKILL.md")
print("=" * 70)
print(content[:1500])  # Display the first part
if len(content) > 1500:
    print(f"\n... ({len(content) - 1500} characters remaining)")


## Step 4: Create the SkillsProvider

`SkillsProvider` is a provider that automatically discovers `SKILL.md` files from a specified directory
and provides skills to the agent.

**Internal operation:**
1. Recursively searches the `skills/` directory (max depth 2)
2. Parses the `SKILL.md` YAML frontmatter (name, description)
3. Validates the existence of referenced resource files
4. Auto-generates 2 tools: `load_skill` / `read_skill_resource`
5. Injects the skill list into the system prompt at session start


In [ ]:
# Create SkillsProvider
skills_provider = SkillsProvider(skill_paths=str(skills_dir))

print("✅ SkillsProvider created successfully")
print()

# Check internal state
print(f"📦 Number of skills discovered: {len(skills_provider._skills)}")
for name, skill in skills_provider._skills.items():
    print(f"\n  🔧 {name}")
    print(f"     Description: {skill.description[:80]}...")
    print(f"     Resources: {[r.name for r in skill.resources]}")
    print(f"     Source: {skill.path}")

print()
print(f"🔧 Number of auto-generated tools: {len(skills_provider._tools)}")
for tool in skills_provider._tools:
    print(f"   - {tool.name}: {tool.description}")


In [ ]:
# Check the content injected into the system prompt
print("=" * 70)
print("📋 Skill advertisements injected into the agent's system prompt:")
print("=" * 70)
print(skills_provider._instructions)


The above is the content injected during the **Advertise phase**.  
Each skill is presented with only its name and description (~100 tokens/skill) using `<skill>` tags.  
The agent looks at this list and calls `load_skill` → `read_skill_resource` as needed.


## Step 5: Run an Agent with Skills (Basic)

Let's ask an agent with skills a question and observe the Progressive Disclosure pattern in action.

What the agent does internally:
1. If the question matches a skill's domain → automatically calls `load_skill("travel-planner")`
2. If detailed FAQ is needed → calls `read_skill_resource("travel-planner", "references/TRAVEL_FAQ.md")`
3. Generates a response based on the retrieved knowledge


In [ ]:
# Create Azure OpenAI client
chat_client = AzureOpenAIChatClient(
    **auth_kwargs,
    endpoint=azure_openai_endpoint,
    api_version=api_version,
    deployment_name=azure_deployment,
)

# Create an agent with skills
# Just pass the skill provider to context_providers!
travel_agent = Agent(
    client=chat_client,
    name="TravelAssistant",
    instructions="You are a helpful travel assistant. Use available skills to answer user questions. Respond in English.",
    context_providers=[skills_provider],
)

print("✅ Agent with skills created successfully")
print(f"   Name: {travel_agent.name}")
print(f"   Skills: {list(skills_provider._skills.keys())}")


In [ ]:
# Example 1: A question that invokes the travel skill
print("=" * 70)
print("💬 Example 1: Travel Planning")
print("=" * 70)

question = "I'm planning a 2-night, 3-day trip to Kyoto. My budget is 50,000 yen. Can you suggest a plan?"
print(f"\n🧑 User: {question}\n")

with tracer.start_as_current_span("SkillDemo-TravelPlan"):
    response = await travel_agent.run(question)
    print(f"🤖 Agent:\n{response}")


In [ ]:
# Example 2: A question that loads FAQ resources
print("=" * 70)
print("💬 Example 2: Travel FAQ Question")
print("=" * 70)

question2 = "Do I need travel insurance for overseas trips? Also, what size suitcase should I get for a 5-night trip?"
print(f"\n🧑 User: {question2}\n")

with tracer.start_as_current_span("SkillDemo-TravelFAQ"):
    response2 = await travel_agent.run(question2)
    print(f"🤖 Agent:\n{response2}")


### Check Traces in Jaeger

When you check traces in Jaeger UI ([http://localhost:16686](http://localhost:16686)),
you can see that the agent calls the following tools in order:

1. `load_skill("travel-planner")` — Retrieve the skill body
2. `read_skill_resource("travel-planner", "references/TRAVEL_FAQ.md")` — Retrieve FAQ (when needed)
3. `read_skill_resource("travel-planner", "assets/itinerary-template.md")` — Retrieve template (when needed)

This is the **Progressive Disclosure** pattern in action.


## Step 6: Try the Code Review Skill

The same skill provider also includes a `code-reviewer` skill.  
The agent automatically selects the appropriate skill based on the question content.


In [ ]:
# Example 3: A question that invokes the code review skill
print("=" * 70)
print("💬 Example 3: Code Review")
print("=" * 70)

code_question = """
Please review the following Python code:

```python
import sqlite3

def get_user(name):
    conn = sqlite3.connect('users.db')
    result = conn.execute(f"SELECT * FROM users WHERE name = '{name}'")
    data = result.fetchall()
    return data

def process_items(items):
    output = []
    for i in range(len(items)):
        output.append(items[i] * 2)
    return output
```
"""
print(f"🧑 User: {code_question}")

with tracer.start_as_current_span("SkillDemo-CodeReview"):
    response3 = await travel_agent.run(code_question)
    print(f"🤖 Agent:\n{response3}")


## Step 7: Leverage Skills in Multi-Turn Conversations

Use sessions for multi-turn conversations and verify that skill knowledge is retained.


In [ ]:
# Multi-turn conversation
print("=" * 70)
print("💬 Multi-Turn Conversation Demo")
print("=" * 70)

session = travel_agent.create_session()

conversation = [
    "I'm planning a 3-night, 4-day trip to Taipei. My budget is 80,000 yen.",
    "Please use the itinerary template to create a specific plan.",
    "Is there anything I should particularly pay attention to regarding what to bring?",
]

with tracer.start_as_current_span("SkillDemo-MultiTurn"):
    for i, user_msg in enumerate(conversation, 1):
        print(f"\n{'─' * 70}")
        print(f"🧑 Turn {i}: {user_msg}")
        print(f"{'─' * 70}\n")
        
        response = await travel_agent.run(user_msg, session=session)
        print(f"🤖 Agent:\n{response}\n")


## Step 8: Inspect the Internal Workings of the SkillsProvider

Let's directly call the internal functions `load_skill` and `read_skill_resource` to inspect their return values.


In [ ]:
# Verify load_skill behavior
print("=" * 70)
print("🔍 Return value of load_skill function:")
print("=" * 70)
body = skills_provider._load_skill("travel-planner")
print(body[:800])
print(f"\n... (total: {len(body)} characters)")

print()

# Verify read_skill_resource behavior
print("=" * 70)
print("🔍 Return value of read_skill_resource function:")
print("=" * 70)
faq = await skills_provider._read_skill_resource("travel-planner", "references/TRAVEL_FAQ.md")
print(faq[:500])
print(f"\n... (total: {len(faq)} characters)")

print()

# When calling a non-existent skill
print("=" * 70)
print("🔍 When calling a non-existent skill:")
print("=" * 70)
error_result = skills_provider._load_skill("nonexistent-skill")
print(f"Result: {error_result}")


## Step 9: How to Create Custom Skills

Let's review the steps for creating a new skill. You can add your own skills by following these steps.


In [ ]:
import tempfile

# Demo: Programmatically create a new skill
print("=" * 70)
print("🛠️ Custom Skill Creation Demo")
print("=" * 70)

# Create a skill in a temporary directory
with tempfile.TemporaryDirectory() as tmpdir:
    skill_dir = Path(tmpdir) / "greeting-helper"
    skill_dir.mkdir()
    
    # Create SKILL.md
    skill_md = skill_dir / "SKILL.md"
    skill_md.write_text("""\
---
name: greeting-helper
description: Teaches greetings in various languages and cultural etiquette. Use for greetings in overseas travel or business settings.
---

# Greeting Helper

## Basic Greetings

| Language | Hello | Thank you | Goodbye |
|------|-----------|-----------|----------|
| English | Hello | Thank you | Goodbye |
| French | Bonjour | Merci | Au revoir |
| Spanish | Hola | Gracias | Adiós |
| Chinese | 你好 | 谢谢 | 再见 |
| Korean | 안녕하세요 | 감사합니다 | 안녕히 가세요 |

## Etiquette

- Japan: The angle of the bow indicates the level of respect
- France: Cheek kisses (la bise) are for close acquaintances
- Thailand: Greet with a wai (palms together)
""", encoding="utf-8")
    
    print(f"📄 Skill file created: {skill_md}")
    print()
    
    # Create provider with the custom skill
    custom_provider = SkillsProvider(skill_paths=tmpdir)
    
    print(f"✅ Custom skill provider created successfully")
    print(f"📦 Skills discovered: {list(custom_provider._skills.keys())}")
    print()
    print("📋 Prompt to be injected:")
    print(custom_provider._instructions)


## Step 10: Granting Skills with the `as_agent` Method

You can also grant skills using `chat_client.as_agent()`.


In [ ]:
# context_providers can also be passed via the as_agent method
agent_via_as_agent = chat_client.as_agent(
    name="SkillfulAgent",
    instructions="You are a multi-functional assistant. Use available skills to answer questions. Respond in English.",
    context_providers=[skills_provider],
)

print("✅ Agent with skills created via as_agent method")
print(f"   Name: {agent_via_as_agent.name}")
print(f"   Number of skills: {len(skills_provider._skills)}")
print()

# Test execution
with tracer.start_as_current_span("SkillDemo-AsAgent"):
    answer = await agent_via_as_agent.run(
        "What are the 3 most important security check items in a Python code review?"
    )
    print(f"🧑 Question: What are the 3 most important security check items in a Python code review?")
    print(f"\n🤖 Answer:\n{answer}")


## Summary

### What We Learned

1. **The Concept of Agent Skills**
   - Modular knowledge packages (`SKILL.md` + resource files)
   - Compliant with the [Agent Skills specification](https://agentskills.io/)
   - Moving away from writing all knowledge in instructions

2. **Progressive Disclosure Pattern**
   - **Advertise**: Inject name + description into system prompt (lightweight)
   - **Load**: Load body on demand with the `load_skill` tool
   - **Read Resources**: Retrieve FAQ/templates with the `read_skill_resource` tool

3. **How to Use SkillsProvider**
   ```python
   provider = SkillsProvider(skill_paths="skills/")
   agent = Agent(
       client=chat_client,
       instructions="...",
       context_providers=[provider],
   )
   ```

4. **Skill Directory Structure**
   ```
   skills/
   └── my-skill/
       ├── SKILL.md           ← YAML frontmatter (name, description) + body
       ├── references/        ← Reference documents
       └── assets/            ← Templates, etc.
   ```

5. **Security**
   - Prompt injection defense via XML escaping
   - Path traversal defense
   - Symbolic link escape defense

### Production Tips

- **Skill names**: Lowercase alphanumeric + hyphens only (e.g., `travel-planner`)
- **Description**: Within 1024 characters, clearly state what tasks it's used for
- **Resources**: Organize FAQ and templates in `references/` and `assets/`
- **Version control**: Manage skill versions with the `metadata` `version` field
- **Testing**: Always test that an agent correctly loads a new skill after creation

---

### Next Steps

- Try creating custom skills tailored to your domain
- Try combining skills with multi-agent workflows (Tutorials 05-07)
- Review the details of the [Agent Skills specification](https://agentskills.io/)


## Exercises

1. **Create Your Own Skill**: Create a `SKILL.md` for your area of expertise (cooking recipes, fitness, programming languages, etc.) and assign it to an agent

2. **Utilize Multiple Resources**: Create a skill with both references and assets, and verify that the agent appropriately uses different resources

3. **Automatic Skill Selection**: Create an agent with 3 or more skills, ask questions from different domains in succession, and verify that the correct skill is selected


In [ ]:
# Exercise: Create and test your own custom skill

async def exercise_custom_skill():
    """
    Exercise: Create your own skill and assign it to an agent
    
    Steps:
    1. Create a new folder in the skills/ directory (e.g., skills/cooking-helper/)
    2. Create SKILL.md (YAML frontmatter + body)
    3. Add files to references/ or assets/ as needed
    4. Load with SkillsProvider
    5. Ask the agent questions to test
    """
    pass

print("🎯 Exercise ready - Try creating your own skill!")
